# DERM-7 — Compute check
Confirms the Kaggle GPU works and the full HAM10000 dataset is mounted.

In [ ]:
# DERM-7: compute check — GPU + HAM10000 available
import glob, torch, pandas as pd

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())

meta_path = glob.glob("/kaggle/input/**/HAM10000_metadata.csv", recursive=True)[0]
df = pd.read_csv(meta_path)
imgs = {p.split("/")[-1] for p in glob.glob("/kaggle/input/**/ISIC_*.jpg", recursive=True)}
print("metadata:", meta_path)
print("rows:", len(df), "| lesions:", df.lesion_id.nunique(), "| unique images on disk:", len(imgs))
print("every metadata image present:", set(df.image_id + ".jpg") <= imgs)

# one training step on the GPU (ResNet-18, batch 32)
import torchvision as tv
dev = "cuda" if torch.cuda.is_available() else "cpu"
m = tv.models.resnet18(weights=None).to(dev)
x = torch.randn(32, 3, 224, 224, device=dev)
m(x).sum().backward()
print("test training step OK on", dev)
